In [ ]:
from datetime import datetime
from datetime import timedelta
from ordered_set import OrderedSet
import numpy as np
from scipy.stats import uniform

from stonesoup.models.transition.linear import CombinedLinearGaussianTransitionModel, \
                                               ConstantVelocity
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.types.detection import TrueDetection
from stonesoup.types.detection import Clutter
from stonesoup.models.measurement.linear import LinearGaussian

np.random.seed(1991)

truths = OrderedSet()
num_steps = 20
start_time = datetime.now().replace(microsecond=0)
transition_model = CombinedLinearGaussianTransitionModel([ConstantVelocity(0.005),
                                                          ConstantVelocity(0.005)])

timesteps = [start_time]
truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])
for k in range(1, num_steps + 1):
    timesteps.append(start_time + timedelta(seconds=k))
    truth.append(GroundTruthState(
        transition_model.function(truth[k-1], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k]))
truths.add(truth)

truth = GroundTruthPath([GroundTruthState([0, 1, 20, -1], timestamp=timesteps[0])])
for k in range(1, num_steps + 1):
    truth.append(GroundTruthState(
        transition_model.function(truth[k-1], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k]))
truths.add(truth)

# Plot ground truth.
from stonesoup.plotter import AnimatedPlotterly
plotter = AnimatedPlotterly(timesteps, tail_length=0.3)
plotter.plot_ground_truths(truths, [0, 2])

# Generate measurements.
all_measurements = []

measurement_model = LinearGaussian(
    ndim_state=4,
    mapping=(0, 2),
    noise_covar=np.array([[0.75, 0],
                          [0, 0.75]])
    )

prob_detect = 0.85  # 90% chance of detection.

for k in range(num_steps):
    measurement_set = set()

    for truth in truths:
        # Generate actual detection from the state with a 10% chance that no detection is received.
        if np.random.rand() <= prob_detect:
            measurement = measurement_model.function(truth[k], noise=True)
            measurement_set.add(TrueDetection(state_vector=measurement,
                                              groundtruth_path=truth,
                                              timestamp=truth[k].timestamp,
                                              measurement_model=measurement_model))

        # Generate clutter at this time-step
        truth_x = truth[k].state_vector[0]
        truth_y = truth[k].state_vector[2]
        for _ in range(np.random.randint(10)):
            x = uniform.rvs(truth_x - 10, 20)
            y = uniform.rvs(truth_y - 10, 20)
            measurement_set.add(Clutter(np.array([[x], [y]]), timestamp=truth[k].timestamp,
                                        measurement_model=measurement_model))
    all_measurements.append(measurement_set)

# Plot true detections and clutter.
plotter.plot_measurements(all_measurements, [0, 2])
plotter.fig

from stonesoup.hypothesiser.probability import PDAHypothesiser

from stonesoup.predictor.kalman import KalmanPredictor
predictor = KalmanPredictor(transition_model)

from stonesoup.updater.kalman import KalmanUpdater
updater = KalmanUpdater(measurement_model)

hypothesiser = PDAHypothesiser(predictor=predictor,
                               updater=updater,
                               clutter_spatial_density=0.25,
                               prob_detect=prob_detect,
                               prob_gate=0.999
                               )

from stonesoup.dataassociator.probability import JPDA
data_associator = JPDA(hypothesiser=hypothesiser)

from stonesoup.types.state import GaussianState
from stonesoup.types.track import Track
from stonesoup.types.array import StateVectors
from stonesoup.functions import gm_reduce_single
from stonesoup.types.update import GaussianStateUpdate

prior1 = GaussianState([[0], [1], [0], [1]], np.diag([1.5, 0.5, 1.5, 0.5]), timestamp=start_time)
prior2 = GaussianState([[0], [1], [20], [-1]], np.diag([1.5, 0.5, 1.5, 0.5]), timestamp=start_time)

tracks = {Track([prior1]), Track([prior2])}

for n, measurements in enumerate(all_measurements):
    hypotheses = data_associator.associate(tracks,
                                           measurements,
                                           start_time + timedelta(seconds=n))

    # Loop through each track, performing the association step with weights adjusted according to
    # JPDA.
    for track in tracks:
        track_hypotheses = hypotheses[track]

        posterior_states = []
        posterior_state_weights = []
        for hypothesis in track_hypotheses:
            if not hypothesis:
                posterior_states.append(hypothesis.prediction)
            else:
                posterior_state = updater.update(hypothesis)
                posterior_states.append(posterior_state)
            posterior_state_weights.append(hypothesis.probability)

        means = StateVectors([state.state_vector for state in posterior_states])
        covars = np.stack([state.covar for state in posterior_states], axis=2)
        weights = np.asarray(posterior_state_weights)

        # Reduce mixture of states to one posterior estimate Gaussian.
        post_mean, post_covar = gm_reduce_single(means, covars, weights)

        # Add a Gaussian state approximation to the track.
        track.append(GaussianStateUpdate(
            post_mean, post_covar,
            track_hypotheses,
            track_hypotheses[0].measurement.timestamp))

plotter.plot_tracks(tracks, [0, 2], uncertainty=True)
plotter.fig

import matplotlib.pyplot as plt
import scienceplots

# 1. Choose an aesthetic style
plt.style.use(['science','no-latex'])
fig, ax = plt.subplots(figsize=(10,7))

# Ground-truths
for i, truth in enumerate(truths, start=1):
    xs = [s.state_vector[0] for s in truth]
    ys = [s.state_vector[2] for s in truth]
    ax.plot(xs, ys, '--o', label=f"Truth {i}")

# Measurements (split clutter vs true)
det_x, det_y, cl_x, cl_y = [], [], [], []
for ms in all_measurements:
    for m in ms:
        x,y = m.state_vector[0,0], m.state_vector[1,0]
        if isinstance(m, TrueDetection):
            det_x.append(x); det_y.append(y)
        else:
            cl_x.append(x); cl_y.append(y)
ax.scatter(cl_x, cl_y, s=12, alpha=0.3, marker='o', label="Clutter")
ax.scatter(det_x, det_y, s=30, alpha=0.8, marker='x', label="True Detections")

# ID-JPDA tracks
for i,tr in enumerate(tracks, start=1):
    xs = [s.state_vector[0,0] for s in tr]
    ys = [s.state_vector[2,0] for s in tr]
    ax.plot(xs, ys, '-s', label=f"JPDA Track {i}")

ax.set_title("StoneSoup JPDA Tracks", pad=15)
ax.set_xlabel("X position")
ax.set_ylabel("Y position")
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
# Compute SIAP Positional Accuracy (RMSE over time)
positional_errors = {f"Track {i+1}": [] for i in range(len(tracks))}
time_labels = []

# Assuming order: truth 1 <-> track 1, truth 2 <-> track 2
for i, (truth, track) in enumerate(zip(truths, sorted(tracks, key=lambda t: t[0].state_vector[2, 0]))):
    for truth_state, track_state in zip(truth[1:], track[1:]):  # Skip initial prior
        gt_pos = np.array([truth_state.state_vector[0], truth_state.state_vector[2]])
        est_pos = np.array([track_state.state_vector[0, 0], track_state.state_vector[2, 0]])
        error = np.linalg.norm(gt_pos - est_pos)
        positional_errors[f"Track {i+1}"].append(error)
    if not time_labels:
        time_labels = [state.timestamp for state in truth[1:]]

# Plotting
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import scienceplots

plt.style.use(['science', 'no-latex'])

fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

# Plot each track's positional error over time
for label, errors in positional_errors.items():
    ax.plot(range(1, len(errors) + 1), errors, label=label,
            marker='o', markersize=4, linewidth=1.8)

# Configure x-axis: 0 to 50, even if data ends at 20
ax.set_xlim(0, num_steps)
ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

# Aesthetics
ax.set_title("SIAP Positional Accuracy (JPDA)", fontsize=14, pad=15)
ax.set_xlabel("Time Step (s)", fontsize=12)
ax.set_ylabel("Position Error (Euclidean distance)", fontsize=12)

ax.grid(True, which='both', linestyle=':', linewidth=0.6, alpha=0.7)
ax.tick_params(axis='both', which='major', labelsize=10)
ax.legend(fontsize=10, loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

# Bar Chart
average_errors = {label: np.mean(errors) for label, errors in positional_errors.items()}

# Plot average positional accuracy as a bar chart
fig, ax = plt.subplots(figsize=(8, 5), dpi=600)

track_labels = list(average_errors.keys())
avg_values = list(average_errors.values())

bars = ax.bar(track_labels, avg_values, color=['tab:orange', 'tab:red'], width=0.5)

# Annotate each bar with its value
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5),  # offset text above bar
                textcoords="offset points",
                ha='center', va='bottom', fontsize=10)

ax.set_title("Average SIAP Positional Accuracy per Track", fontsize=13, pad=10)
ax.set_ylabel("Average Position Error", fontsize=11)
ax.set_ylim(0, max(avg_values) + 5)  # Add some space above bars
ax.grid(axis='y', linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
plt.show()




In [ ]:
# Compute velocity accuracy (Euclidean velocity error per time step)
velocity_errors = {f"Track {i+1}": [] for i in range(len(tracks))}
time_labels_vel = []

# Match ground truths and tracks (assume sorted order)
for i, (truth, track) in enumerate(zip(truths, sorted(tracks, key=lambda t: t[0].state_vector[2, 0]))):
    for truth_state, track_state in zip(truth[1:], track[1:]):  # Skip initial
        gt_vel = np.array([truth_state.state_vector[1], truth_state.state_vector[3]])
        est_vel = np.array([track_state.state_vector[1, 0], track_state.state_vector[3, 0]])
        error = np.linalg.norm(gt_vel - est_vel)
        velocity_errors[f"Track {i+1}"].append(error)
    if not time_labels_vel:
        time_labels_vel = [state.timestamp for state in truth[1:]]

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.style.use(['science', 'no-latex'])

fig, ax = plt.subplots(figsize=(10, 6), dpi=600)

for label, errors in velocity_errors.items():
    ax.plot(range(1, len(errors) + 1), errors, label=label,
            marker='o', markersize=4, linewidth=1.8)

ax.set_xlim(0, num_steps)
ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

ax.set_title("SIAP Velocity Accuracy (JPDA)", fontsize=14, pad=15)
ax.set_xlabel("Time Step (s)", fontsize=12)
ax.set_ylabel("Velocity Error (Euclidean)", fontsize=12)

ax.grid(True, which='both', linestyle=':', linewidth=0.6, alpha=0.7)
ax.tick_params(axis='both', which='major', labelsize=10)
ax.legend(fontsize=10, loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

# Compute average velocity error
avg_vel_errors = {label: np.mean(errors) for label, errors in velocity_errors.items()}

fig, ax = plt.subplots(figsize=(8, 5), dpi=600)

track_labels = list(avg_vel_errors.keys())
avg_values = list(avg_vel_errors.values())

bars = ax.bar(track_labels, avg_values, color=['tab:blue', 'tab:green'], width=0.5)

# Annotate bars
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5), textcoords="offset points",
                ha='center', va='bottom', fontsize=10)

ax.set_title("Average SIAP Velocity Accuracy per Track", fontsize=13, pad=10)
ax.set_ylabel("Avg. Velocity Error (Euclidean)", fontsize=11)
ax.set_ylim(0, max(avg_values) + 1)
ax.grid(axis='y', linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
from stonesoup.metricgenerator.tracktotruthmetrics import SIAPMetrics
from stonesoup.measures import Euclidean
from stonesoup.metricgenerator.manager import MultiManager
from stonesoup.dataassociator.tracktotrack import TrackToTruth

# Set up SIAP metrics generator and associator
associator = TrackToTruth(association_threshold=30)
siap_metrics = SIAPMetrics(
    position_measure=Euclidean((0, 2)),
    velocity_measure=Euclidean((1, 3)),
    generator_name='SIAP',
    tracks_key='tracks',
    truths_key='truths'
)
metric_manager = MultiManager([siap_metrics], associator=associator)

# Add data (tracks and truths)
metric_manager.add_data({'tracks': tracks}, overwrite=False)
metric_manager.add_data({'truths': truths}, overwrite=False)

# Generate metrics
metrics = metric_manager.generate_metrics()
siap_result = metrics['SIAP']

# Extract SIAP metrics
spuriousness_avg = siap_result.get('SIAP Spuriousness').value
ambiguity_avg = siap_result.get('SIAP Ambiguity').value
spuriousness_at_times = [m.value for m in siap_result.get('SIAP Spuriousness at times').value]
ambiguity_at_times = [m.value for m in siap_result.get('SIAP Ambiguity at times').value]

print("SIAP Spuriousness (average):", spuriousness_avg)
print("SIAP Spuriousness at times:", spuriousness_at_times)
print("SIAP Ambiguity (average):", ambiguity_avg)
print("SIAP Ambiguity at times:", ambiguity_at_times)


In [14]:
import numpy as np
from datetime import datetime, timedelta
from ordered_set import OrderedSet
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.transition.linear import CombinedLinearGaussianTransitionModel, ConstantVelocity
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.types.detection import TrueDetection, Clutter
from stonesoup.types.state import GaussianState
from stonesoup.types.track import Track
from stonesoup.types.array import StateVectors
from stonesoup.functions import gm_reduce_single
from stonesoup.types.update import GaussianStateUpdate

from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.updater.kalman import KalmanUpdater
from stonesoup.hypothesiser.probability import PDAHypothesiser
from stonesoup.dataassociator.probability import JPDAwithLBP as JPDA
from stonesoup.initiator.simple import MultiMeasurementInitiator
from stonesoup.deleter.time import UpdateTimeDeleter
from stonesoup.tracker.simple import MultiTargetTracker

    # 1. Create ground truth
def generate_siap_metrics(seed, num_steps):
    np.random.seed(seed)
    start_time = datetime.now().replace(microsecond=0)
    timesteps = [start_time + timedelta(seconds=k) for k in range(num_steps + 1)]

    transition_model = CombinedLinearGaussianTransitionModel([ConstantVelocity(0.005), ConstantVelocity(0.005)])

    truths = OrderedSet()
    truth1 = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])
    for k in range(1, num_steps + 1):
        truth1.append(GroundTruthState(
            transition_model.function(truth1[k-1], noise=True, time_interval=timedelta(seconds=1)),
            timestamp=timesteps[k]))
    truths.add(truth1)

    truth2 = GroundTruthPath([GroundTruthState([0, 1, 20, -1], timestamp=timesteps[0])])
    for k in range(1, num_steps + 1):
        truth2.append(GroundTruthState(
            transition_model.function(truth2[k-1], noise=True, time_interval=timedelta(seconds=1)),
            timestamp=timesteps[k]))
    truths.add(truth2)

    # 2. Generate measurements (with clutter)
    all_measurements = []
    measurement_model = LinearGaussian(
        ndim_state=4,
        mapping=(0, 2),
        noise_covar=np.array([[0.75, 0],
                            [0, 0.75]])
        )
    prob_detect = 0.85

    for k in range(num_steps + 1):
        measurement_set = set()
        for truth in truths:
            # True detection with prob_detect
            if np.random.rand() <= prob_detect:
                measurement = measurement_model.function(truth[k], noise=True)
                measurement_set.add(TrueDetection(
                    state_vector=measurement,
                    groundtruth_path=truth,
                    timestamp=truth[k].timestamp,
                    measurement_model=measurement_model))
            # Add random clutter (simulate "hard" tracking scenario)
            for _ in range(np.random.randint(2)):
                x = np.random.uniform(-10, 30)
                y = np.random.uniform(-10, 30)
                measurement_set.add(Clutter(
                    np.array([[x], [y]]),
                    timestamp=truth[k].timestamp,
                    measurement_model=measurement_model
                ))
        all_measurements.append(measurement_set)

    # 3. Create tracking components (JPDA)
    predictor = KalmanPredictor(transition_model)
    updater = KalmanUpdater(measurement_model)
    hypothesiser = PDAHypothesiser(
        predictor=predictor,
        updater=updater,
        clutter_spatial_density=0.04,
        prob_detect=prob_detect,
        prob_gate=0.999
    )
    data_associator = JPDA(hypothesiser=hypothesiser)

    # Initiator: needs a prior (wide covar) and a deleter for tracks
    prior_state = GaussianState([[0], [1], [0], [1]], np.diag([100, 25, 100, 25]), timestamp=start_time)
    deleter = UpdateTimeDeleter(time_since_update=timedelta(seconds=20))
    initiator = MultiMeasurementInitiator(
        prior_state=prior_state,
        deleter=deleter,
        data_associator=data_associator,
        updater=updater,
        measurement_model=measurement_model,
        min_points=1
    )

    detector = iter(list(zip(timesteps, all_measurements)))
    # 4. Tracker
    tracker = MultiTargetTracker(
        initiator=initiator,
        deleter=deleter,
        data_associator=data_associator,
        updater=updater,
        detector=detector
    )

    # 5. Run tracking
    tracks = set()
    for time, curr_tracks in tracker:
        tracks |= curr_tracks

    # Filter out very short tracks (common in SIAP metric evaluations)
    tracks = {track for track in tracks if len(track) > 3}
    tracks = set(tracks)

    # 6. Evaluate SIAP metrics (optional, if you want, requires stonesoup.metricgenerator)


    from stonesoup.metricgenerator.tracktotruthmetrics import SIAPMetrics
    from stonesoup.measures import Euclidean
    from stonesoup.metricgenerator.manager import MultiManager
    from stonesoup.dataassociator.tracktotrack import TrackToTruth

    associator_eval = TrackToTruth(association_threshold=30)

    siap_metrics = SIAPMetrics(
            position_measure=Euclidean((0,2)),
            velocity_measure=Euclidean((1,3)),
            generator_name='SIAP',
            tracks_key='tracks',
            truths_key='truths'
        )
    metric_manager = MultiManager([siap_metrics], associator=associator_eval)
    metric_manager.add_data({'tracks': tracks}, overwrite=False)
    metric_manager.add_data({'truths': truths}, overwrite=False)
    metrics = metric_manager.generate_metrics()
    siap_result = metrics['SIAP']

    spuriousness = siap_result.get('SIAP Spuriousness').value
    ambiguity = siap_result.get('SIAP Ambiguity').value
    completeness = siap_result.get('SIAP Completeness').value
    position_accuracy = siap_result.get('SIAP Position Accuracy').value
    velocity_accuracy = siap_result.get('SIAP Velocity Accuracy').value
    rate_track_change = siap_result.get('SIAP Rate of Track Number Change').value
    longest_track = siap_result.get('SIAP Longest Track Segment').value
    spuriousness_at_times = siap_result.get('SIAP Spuriousness at times').value
    ambiguity_at_times = siap_result.get('SIAP Ambiguity at times').value
    completeness_at_times = siap_result.get('SIAP Completeness at times').value
    position_accuracy_at_times = siap_result.get('SIAP Position Accuracy at times').value
    velocity_accuracy_at_times = siap_result.get('SIAP Velocity Accuracy at times').value
    return spuriousness, ambiguity, completeness, position_accuracy, velocity_accuracy, rate_track_change, longest_track,\
            spuriousness_at_times, ambiguity_at_times, completeness_at_times, position_accuracy_at_times, velocity_accuracy_at_times

_ = generate_siap_metrics(1, 100)


8.698720838686624
